# installs

In [ ]:
# import os
# import platform
# import subprocess
# import sys
# !pip install kagglehub

# subprocess.check_call([sys.executable, "-m", "pip", "install", "pip3-autoremove"])
# if platform.system() == "Darwin":
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio"])
# else:
#     subprocess.check_call([
#         sys.executable, "-m", "pip", "install",
#         "torch", "torchvision", "torchaudio", "xformers",
#         "--index-url", "https://download.pytorch.org/whl/cu128",
#     ])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "unsloth"])
# # subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers==4.55.4"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-deps", "trl==0.22.2"])

In [ ]:
# %pip install --upgrade "transformers==5.5.0"

# imports

In [ ]:
import ctypes
import os
import pandas as pd
import numpy as np
import re
import multiprocessing
from time import time as timer
from tqdm import tqdm
from pathlib import Path
from functools import partial
import requests
import urllib
from PIL import Image

for cuda_driver_path in ("/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so.1"):
    if Path(cuda_driver_path).exists():
        ctypes.CDLL(cuda_driver_path, mode=ctypes.RTLD_GLOBAL)
        break

from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch
from transformers import TextStreamer

from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig


# download dataest

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("raghavdharwal/amazon-ml-challenge-2025")

print("Path to dataset files:", path)

# extract train subset

In [ ]:
# Load only the training data
train_files = list(Path(path).rglob("train.csv"))
if not train_files:
    raise FileNotFoundError(f"train.csv not found under {path}")

train_df = pd.read_csv(train_files[0])
print(f"Train data shape: {train_df.shape}")
display(train_df.head())


# img download

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

IMAGE_FOLDER = (Path.cwd() / "images").resolve()
IMAGE_FOLDER.mkdir(parents=True, exist_ok=True)

def sample_id_key(sample_id):
    value = str(sample_id)
    return value[:-2] if value.endswith(".0") else value

def download_image(image_link, sample_id, image_folder):
    if not isinstance(image_link, str) or not image_link.strip():
        return sample_id, False, "missing image_link"

    url_path = Path(urlparse(image_link).path)
    extension = url_path.suffix.lower() or ".jpg"
    image_path = image_folder / f"{sample_id_key(sample_id)}{extension}"

    if image_path.exists() and image_path.stat().st_size > 0:
        return sample_id, True, "already exists"

    last_error = "download failed"
    for _ in range(3):
        try:
            response = requests.get(
                image_link,
                timeout=30,
                headers={"User-Agent": "Mozilla/5.0"},
            )
            response.raise_for_status()
            if not response.content:
                raise ValueError("empty response")
            image_path.write_bytes(response.content)
            return sample_id, True, "downloaded"
        except Exception as error:
            last_error = str(error)
    return sample_id, False, last_error

download_rows = list(train_df[["sample_id", "image_link"]].itertuples(index=False))
MAX_WORKERS = 64

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    download_results = list(tqdm(
        executor.map(
            lambda row: download_image(row.image_link, row.sample_id, IMAGE_FOLDER),
            download_rows,
        ),
        total=len(download_rows),
    ))

failed_downloads = [result for result in download_results if not result[1]]
print(f"Images available in {IMAGE_FOLDER.resolve()}: {len(download_results) - len(failed_downloads)}")
print(f"Failed downloads: {len(failed_downloads)}")
if failed_downloads:
    display(pd.DataFrame(failed_downloads, columns=["sample_id", "success", "error"]))

image_paths_by_sample = {
    image_path.stem: image_path
    for image_path in IMAGE_FOLDER.iterdir()
    if image_path.is_file() and image_path.stat().st_size > 0
}
print(f"Indexed local image files: {len(image_paths_by_sample)}")

image_available = train_df["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
train_df_with_images = train_df.loc[image_available].copy()
missing_image_rows = train_df.loc[~image_available, ["sample_id", "image_link"]]
print(f"Rows retained with local images: {len(train_df_with_images)} / {len(train_df)}")
if not missing_image_rows.empty:
    print(f"Rows skipped because their image download failed: {len(missing_image_rows)}")
    display(missing_image_rows.head())

# split 

In [ ]:
from sklearn.model_selection import train_test_split

# Rebuild the source rows from the image index so stale splits cannot include failed downloads
image_available = train_df["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
train_df_with_images = train_df.loc[image_available].copy()
omitted_rows = train_df.loc[~image_available, ["sample_id", "image_link"]]
print(f"Omitting rows without local images: {len(omitted_rows)}")

train, test = train_test_split(
    train_df_with_images,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

print(f"Train split shape: {train.shape}")
print(f"Test split shape: {test.shape}")

# VLM loader + attaching LoRA Adapters

In [ ]:
# Load Qwen2-VL, attach LoRA adapters, and cap the visual token budget.
# NOTE: this cell now runs BEFORE the data-prep cell, because prep needs the tokenizer.
if not torch.cuda.is_available():
    raise RuntimeError("Qwen2-VL fine-tuning requires a CUDA GPU runtime")

MODEL_NAME = "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
USE_GRADIENT_CHECKPOINTING = GPU_MEMORY_GB < 24

model, processor = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth" if USE_GRADIENT_CHECKPOINTING else False,
)
tokenizer = processor                                     # alias: `tokenizer` is really the processor
text_tokenizer = getattr(processor, "tokenizer", processor)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)

# ---- Cap how many tokens one image may consume -------------------------------
# Qwen2-VL emits one token per (patch_size * merge_size)^2 = 28x28 pixel block.
# Depending on the transformers version, min_pixels/max_pixels are either plain
# attributes or read-only properties over `image_processor.size`, and `size` is
# either a dict or a SizeDict dataclass. Handle all four combinations.
MAX_IMAGE_TOKENS = 256
MIN_IMAGE_TOKENS = 64

def cap_visual_tokens(processor, max_image_tokens=MAX_IMAGE_TOKENS, min_image_tokens=MIN_IMAGE_TOKENS):
    image_processor = getattr(processor, "image_processor", processor)
    factor = getattr(image_processor, "patch_size", 14) * getattr(image_processor, "merge_size", 2)
    min_pixels = min_image_tokens * factor ** 2
    max_pixels = max_image_tokens * factor ** 2

    size = getattr(image_processor, "size", None)
    if isinstance(size, dict):                                  # dict / dict subclass
        size.update({"shortest_edge": min_pixels, "longest_edge": max_pixels})
    elif size is not None and hasattr(size, "shortest_edge"):   # SizeDict dataclass
        size.shortest_edge = min_pixels
        size.longest_edge = max_pixels
    else:
        image_processor.size = {"shortest_edge": min_pixels, "longest_edge": max_pixels}

    for attribute, value in (("min_pixels", min_pixels), ("max_pixels", max_pixels)):
        try:
            setattr(image_processor, attribute, value)
        except AttributeError:
            pass          # read-only property on this version; `size` above is the real store
    return image_processor, min_pixels, max_pixels

image_processor, MIN_PIXELS, MAX_PIXELS = cap_visual_tokens(processor)

# Prove the cap actually binds: a deliberately huge image must come back <= MAX_IMAGE_TOKENS.
probe = processor.image_processor(images=Image.new("RGB", (2048, 2048), "white"), return_tensors="pt")
probe_tokens = int(np.prod(probe["image_grid_thw"][0].tolist()) // (getattr(image_processor, "merge_size", 2) ** 2))
print(f"Loaded {MODEL_NAME}; GPU memory: {GPU_MEMORY_GB:.1f} GB; gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}")
print(f"Image budget: min_pixels={MIN_PIXELS}, max_pixels={MAX_PIXELS}")
print(f"2048x2048 probe image -> {probe_tokens} image tokens (cap is {MAX_IMAGE_TOKENS})")
assert probe_tokens <= MAX_IMAGE_TOKENS, "Image token cap did not take effect"

# prep

In [ ]:
# Prepare multimodal conversations for Qwen2-VL.
# Two changes vs. the original, both aimed at the image-token mismatch error:
#   1. the image comes FIRST in the content list, so right-side truncation can
#      never eat the vision block;
#   2. the catalog text is clipped to a token budget that leaves room for the
#      image, the instruction and the answer inside MAX_SEQ_LENGTH.
train_image_available = train["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
test_image_available = test["sample_id"].map(sample_id_key).isin(image_paths_by_sample)
omitted_train_count = int((~train_image_available).sum())
omitted_test_count = int((~test_image_available).sum())
train = train.loc[train_image_available].copy()
test = test.loc[test_image_available].copy()
print(f"Omitted rows without local images before preparation: train={omitted_train_count}, test={omitted_test_count}")

TEXT_COLUMN = next((column for column in ("Content_catalougr", "catalog_content") if column in train.columns), None)
if TEXT_COLUMN is None:
    raise KeyError("Expected Content_catalougr or catalog_content in the dataset")
if "price" not in train.columns or "price" not in test.columns:
    raise KeyError("The train/test splits must contain the price column")

PRICE_INSTRUCTION = """
You are a product pricing assistant. Predict the product price in USD using the catalog text and product image.
Do not treat weights, volumes, quantities, years, or model numbers as the price.
Return only one positive numeric price, without currency symbols or explanation.
"""

# ---- Token budget ------------------------------------------------------------
MAX_SEQ_LENGTH = 1024
CHAT_TEMPLATE_AND_ANSWER_TOKENS = 64          # <|im_start|> wrappers, role tags, the price itself
INSTRUCTION_TOKENS = len(text_tokenizer.encode(PRICE_INSTRUCTION, add_special_tokens=False))
MAX_TEXT_TOKENS = MAX_SEQ_LENGTH - MAX_IMAGE_TOKENS - INSTRUCTION_TOKENS - CHAT_TEMPLATE_AND_ANSWER_TOKENS
if MAX_TEXT_TOKENS < 64:
    raise ValueError("MAX_SEQ_LENGTH is too small for the image budget plus the instruction")
print(
    f"Budget -> seq {MAX_SEQ_LENGTH} = image {MAX_IMAGE_TOKENS} + instruction {INSTRUCTION_TOKENS} "
    f"+ catalog {MAX_TEXT_TOKENS} + overhead {CHAT_TEMPLATE_AND_ANSWER_TOKENS}"
)

def clip_text(text, max_tokens=MAX_TEXT_TOKENS):
    text = "" if text is None or (isinstance(text, float) and np.isnan(text)) else str(text)
    if len(text) <= max_tokens:          # 1 token is always >= 1 character, so this is safe
        return text
    ids = text_tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text
    return text_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

def image_path_for_sample(sample_id):
    sample_key = sample_id_key(sample_id)
    if sample_key not in image_paths_by_sample:
        raise FileNotFoundError(f"No downloaded image found for sample_id={sample_id}")
    return image_paths_by_sample[sample_key]

def load_sample_image(sample_id):
    image_path = image_path_for_sample(sample_id)
    with Image.open(image_path) as image:
        return image.convert("RGB")

def build_user_content(row):
    return [
        {"type": "image", "image": str(image_path_for_sample(row["sample_id"]))},
        {"type": "text", "text": f"{PRICE_INSTRUCTION}\n\n{clip_text(row[TEXT_COLUMN])}"},
    ]

def convert_row_to_conversation(row, include_answer=True):
    messages = [{"role": "user", "content": build_user_content(row)}]
    if include_answer:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": f"{float(row['price']):.2f}"}],
        })
    return {"messages": messages}

train_dataset = [
    convert_row_to_conversation(row)
    for row in tqdm(train.to_dict(orient="records"), desc="Preparing training examples")
]
print(f"Using text column: {TEXT_COLUMN}")
print(f"Prepared training examples: {len(train_dataset)}")

# sanity check (new)

Runs the real collator over 64 samples. If anything still gets truncated, it fails here in seconds instead of mid-training.

In [ ]:
# Sanity check: confirm NO sample needs truncation before spending a GPU-hour on it.
# If the image block were still being clipped, the collator raises the
# "Mismatch in `image` token count" ValueError right here, on 64 samples.
import random

IMAGE_TOKEN_ID = getattr(processor, "image_token_id", None)
if IMAGE_TOKEN_ID is None:
    IMAGE_TOKEN_ID = text_tokenizer.convert_tokens_to_ids("<|image_pad|>")

probe_collator = UnslothVisionDataCollator(model, processor, max_seq_length=MAX_SEQ_LENGTH)
probe_samples = random.Random(42).sample(train_dataset, min(64, len(train_dataset)))

sequence_lengths, image_token_counts = [], []
for start in range(0, len(probe_samples), 4):
    batch = probe_collator(probe_samples[start:start + 4])
    input_ids = batch["input_ids"]
    sequence_lengths.append(int(input_ids.shape[-1]))
    image_token_counts.extend((input_ids == IMAGE_TOKEN_ID).sum(-1).tolist())

print(f"Checked {len(probe_samples)} samples")
print(f"Sequence length: max={max(sequence_lengths)} (limit {MAX_SEQ_LENGTH})")
print(f"Image tokens per sample: min={min(image_token_counts)}, max={max(image_token_counts)}")
assert max(sequence_lengths) <= MAX_SEQ_LENGTH, "Some sample still exceeds MAX_SEQ_LENGTH"
assert min(image_token_counts) > 0, "Some sample lost its image block"
assert max(image_token_counts) <= MAX_IMAGE_TOKENS, "Some image exceeds the token cap"
del probe_collator, probe_samples
print("OK - no truncation required, every sample keeps its full image.")

# Fine-tuning

In [ ]:
# Fine-tune on the 80% training split.
# GPU_MEMORY_GB is taken from the loader cell now; the original hard-coded 80
# forced micro-batch 8 on every machine regardless of the actual card.
EFFECTIVE_BATCH_SIZE = 8
if GPU_MEMORY_GB >= 48:
    PER_DEVICE_TRAIN_BATCH_SIZE = 8
elif GPU_MEMORY_GB >= 24:
    PER_DEVICE_TRAIN_BATCH_SIZE = 4
elif GPU_MEMORY_GB >= 12:
    PER_DEVICE_TRAIN_BATCH_SIZE = 2
else:
    PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = max(1, EFFECTIVE_BATCH_SIZE // PER_DEVICE_TRAIN_BATCH_SIZE)
DATALOADER_NUM_WORKERS = 2
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_TF32 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    data_collator=UnslothVisionDataCollator(
        model,
        processor,
        max_seq_length=MAX_SEQ_LENGTH,     # safety net only; the budget above keeps us under it
    ),
    train_dataset=train_dataset,
    args=SFTConfig(
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="qwen2vl_price_training",
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQ_LENGTH,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        bf16=USE_BF16,
        fp16=not USE_BF16,
        tf32=USE_TF32,
    ),
)
print(
    f"GPU memory: {GPU_MEMORY_GB:.1f} GB; "
    f"micro-batch: {PER_DEVICE_TRAIN_BATCH_SIZE}; "
    f"gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}; "
    f"effective batch: {EFFECTIVE_BATCH_SIZE}"
)
trainer_stats = trainer.train()
print(trainer_stats.metrics)

# Testing

In [ ]:
# Generate predictions for the 20% test split and calculate SMAPE.
# The prompt is built with the same helper as training (image first, clipped text)
# so evaluation matches the training distribution.
FastVisionModel.for_inference(model)

def extract_price(text):
    match = re.search(r"(?<!\d)(\d+(?:\.\d+)?)(?!\d)", text.replace(",", ""))
    if match is None:
        return np.nan
    value = float(match.group(1))
    return value if np.isfinite(value) and value > 0 else np.nan

def predict_price(row):
    image = load_sample_image(row["sample_id"])
    content = build_user_content(row)
    content[0] = {"type": "image", "image": image}        # pass the PIL object, not the path
    messages = [{"role": "user", "content": content}]

    prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(
        images=[image],
        text=[prompt],
        add_special_tokens=False,      # apply_chat_template already added them
        return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            use_cache=True,
        )
    prompt_length = inputs["input_ids"].shape[-1]
    generated_text = text_tokenizer.decode(
        output_ids[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()
    return generated_text, extract_price(generated_text)

raw_predictions = []
predicted_prices = []
for row in tqdm(test.to_dict(orient="records"), desc="Evaluating test split"):
    raw_prediction, predicted_price = predict_price(row)
    raw_predictions.append(raw_prediction)
    predicted_prices.append(predicted_price)

test_results = test[["sample_id", "price"]].copy()
test_results["raw_prediction"] = raw_predictions
test_results["predicted_price"] = predicted_prices
fallback_price = float(train["price"].median())
test_results["predicted_price"] = test_results["predicted_price"].fillna(fallback_price).clip(lower=0.01)

def smape(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.abs(actual) + np.abs(predicted)
    values = np.where(denominator == 0, 0.0, 2.0 * np.abs(predicted - actual) / denominator)
    return 100.0 * values.mean()

invalid_count = int(pd.isna(predicted_prices).sum())
score = smape(test_results["price"], test_results["predicted_price"])
print(f"Invalid generated prices replaced with median: {invalid_count}")
print(f"Test SMAPE: {score:.4f}%")
display(test_results.head())

OUTPUT_DIR = Path("qwen2vl_price_lora")
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
test_results.to_csv("qwen2vl_test_predictions.csv", index=False)
print(f"Saved LoRA adapter and processor to {OUTPUT_DIR.resolve()}")
print("Saved predictions to qwen2vl_test_predictions.csv")